In [7]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1️⃣ Load dataset
with open("cricketers.json", "r", encoding="utf-8") as f:
    characters = json.load(f)

descriptions = [c["description"] for c in characters]

print("Loaded", len(descriptions), "characters")

# 2️⃣ Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 3️⃣ Generate embeddings
embeddings = model.encode(descriptions, convert_to_numpy=True)

# 4️⃣ Normalize vectors (important for cosine similarity)
faiss.normalize_L2(embeddings)

# 5️⃣ Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # inner product

# 6️⃣ Add embeddings to index
index.add(embeddings)

# 7️⃣ Save index
faiss.write_index(index, "cricketers_index.faiss")

print("Index built successfully!")

Loaded 119 characters


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 849.07it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index built successfully!


In [ ]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load data
with open("cricketers.json", "r", encoding="utf-8") as f:
    characters = json.load(f)

# Load FAISS index
index = faiss.read_index("cricketers_index.faiss")

# Load model
model = SentenceTransformer("all-MiniLM-L6-v2")

while True:
    query = input("\nEnter description (or 'exit'): ")

    if query.lower() == "exit":
        break

    query_vector = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vector)

    k = 3
    distances, indices = index.search(query_vector, k)

    print("\nTop Matches:")
    for i, idx in enumerate(indices[0]):
        print(f"{i+1}. {characters[idx]['name']} (Score: {distances[0][i]:.3f})")